# Notebook_C (dùng chung, phần dữ liệu, Sinh viên A): thu thập, so sánh nguồn, làm sạch, EDA và SQL

**Dùng chung cho mọi đề tài của môn.** Chỉ cần điền khối CONFIG bên dưới (tên đề tài, nguồn, tên cột, tần suất) rồi chạy tuần tự. Mọi ô code có chú thích tiếng Anh; phần giải thích, lý do và cách đọc kết quả bằng tiếng Việt nằm ngay trên mỗi ô.

**Sản phẩm bàn giao cho Sinh viên B**: `data/processed/feat.parquet`, `data/processed/manifest.json`, thư mục `report/` (bảng RQ1, hình, nhật ký làm sạch, AI Audit Log).

**Ba câu hỏi nghiên cứu chung (áp dụng cho mọi đề, mỗi nhóm thêm 1-2 câu riêng theo đề):**

* **RQ1 (dữ liệu và mô tả, sâu hơn)**: Hai nguồn dữ liệu (chính và phụ) có nhất quán với nhau không, biến mục tiêu biến động theo mùa, theo đơn vị và theo biến phụ như thế nào, và phần nào của biến động được giải thích bằng lịch so với bằng nguồn phụ?
* **RQ2 (mô hình và tổng quát hoá)**: Trong sáu họ mô hình (mốc naive, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot), mô hình nào tốt nhất theo tầm dự báo, và kết quả có giữ được khi lùi gốc dự báo theo thời gian và khi gặp đơn vị chưa thấy không?
* **RQ3 (tin cậy và vận hành)**: Khoảng tin cậy có đúng bao phủ không, cảnh báo sự kiện (mục tiêu vượt phân vị cao) đạt recall và precision bao nhiêu, sai số thay đổi thế nào khi chế độ đổi, và nguồn phụ đóng góp bao nhiêu (ablation)?

Notebook_C trả lời RQ1 và chuẩn bị dữ liệu cho RQ2, RQ3 (Notebook_D).

In [120]:
# CONFIG: fill in once for your group, both notebooks read the same block
TOPIC = "Multimodal Classification and Fast Small-model Summarisation of UAS Incident Reports from NASA ASRS for Safe Aviation Infrastructure"
GROUP = "Nhom 6"                     # group label used in file names and the audit log
PRIMARY_SOURCE = {"name": "Aviation Safety Reporting System", "url": "asrs.arc.nasa.gov", "license": "public domain","path": "../data/raw/ASRS_FULL_DATASET.csv"}
SECOND_SOURCE  = {"name": "NTSB Aviation Accident/Incident Database (avall.zip)", "url": "https://data.ntsb.gov/avdata/", "license": "public domain","path": "../data/raw/ntsb_exog_merged.csv"}
UNIT_COL   = "unit"     # mã sân bay thật — ASRS: rút từ "Place" (loại ZZZ/ZZZZ); NTSB: cột "ev_nr_apt_id"
TIME_COL   = "ts"       # mốc thời gian theo quý
TARGET_COL = "y"        # số lượng báo cáo ASRS trong mỗi (unit, quý) — tự đếm bằng groupby().size()
EXOG_COLS  = ["exog_human_factors", "exog_maintenance", "exog_weather", "exog_environmental"]  # đếm số finding NTSB theo 4 nhóm, join theo (unit, ts)
FREQ       = "Q"        # quý — khớp độ phân giải thật của cả 2 nguồn
HORIZONS   = [4, 5]     # lag 4 và 5 quý — theo cấu trúc đã có ý nghĩa thống kê trong Heiner (2025)
SEASON     = 4          # 4 quý/năm
DATE_START = "2005-01-01"   # phạm vi thời gian cố định của đề tài
DATE_END   = "2025-12-31"
TEST_START = "2024-01-01"   # first timestamp of the test period (time-based split)
EVENT_QUANTILE = 0.9        # "event" = target above this quantile (used by RQ3 warning metrics)

print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Multimodal Classification and Fast Small-model Summarisation of UAS Incident Reports from NASA ASRS for Safe Aviation Infrastructure
Group: Nhom 6


In [121]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("../report", exist_ok=True)
AUDIT_PATH = "../report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")
audit(
    step="Tim nguon phu (weather/airport)",
    prompt="Đề tài của tôi là {Multimodal Classification and Fast Small-model Summarisation of UAS Incident Reports from NASA ASRS for Safe Aviation Infrastructure}(cái tôi đã gửi). Nguồn chính là {asrs.arc.nasa.gov}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết , sân bay , các phương tiện , vv ) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép.",
    tool="Claude",
    ai_output_summary="De xuat NOAA/Iowa Environmental Mesonet - ASOS/METAR, OurAirports, BTS/FAA - Air Traffic Activity System (ATADS) hoặc OpenSky Network",
    verified_how="Kiem tra URL con hoat dong bang web search",  
    decision="Tam giu IEM/OurAirports, sau doi ATADS -> NTSB",
    hallucination=False
)

audit(
    step="So sánh 2 nguồn ASRS và NTSB",
    prompt="4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn: ASRS và NTSB. Hãy phân tích xem có thể dùng NTSB làm SECOND_SOURCE thay cho ATADS không.",
    tool="Claude",
    ai_output_summary="De xuat 4 chi so: so unit chung, khoang thoi gian, so dong join duoc, tuong quan Pearson",
    verified_how="Doc noi dung PDF, kiem tra phan Data availability statement",
    decision="Khong dua vao CONFIG, chi ghi chu de trich dan trong Related Work",
    hallucination=False
)

audit(
    step="Xac dinh SECOND_SOURCE = NTSB",
    prompt="Doc luan an Heiner (2025) va danh gia dung duoc khong; sau do huong dan lay du lieu tu data.ntsb.gov",
    tool="Claude",
    ai_output_summary="Luan an dung NTSB+ASRS theo quy, lag 4-5 quy co y nghia thong ke; huong dan tai avall.zip, mo bang Access, export events va Findings",
    verified_how="Tu mo Microsoft Access, kiem tra ten bang that (events, Findings, khong phai dt_events), doc thu bang pandas",
    decision="Dung events.csv + finding.csv lam SECOND_SOURCE thay vi ATADS",
    hallucination=False
)

audit(
    step="Sua CONFIG - UNIT_COL/TIME_COL/TARGET_COL",
    prompt="Sua code lai dum toi voi cau truc that cua ASRS (Place bi an danh ZZZ, khong co ngay cu the)",
    tool="Claude",
    ai_output_summary="UNIT_COL rut tu Place loai ZZZ; TIME_COL gop thang+khung 6h roi quy tron ve quy; TARGET_COL tu dem bang groupby().size()",
    verified_how="Chay df['Place'].value_counts() xac nhan ty le ZZZ ~54%",
    decision="Ap dung dung nhu de xuat, doi FREQ tu 'h' sang 'Q', HORIZONS=[4,5]",
    hallucination=False
)

audit(
    step="Xay EXOG_COLS tu bang Findings (NTSB)",
    prompt="Sau khi co events.csv va finding.csv, xu ly the nao de ra 4 cot EXOG_COLS",
    tool="Claude",
    ai_output_summary="Map category_no: 1=maintenance, 2=human_factors, 3=weather/environmental (tach theo tu khoa 'weather'), join qua ev_id, dem theo (unit, ts)",
    verified_how="Chay fd.groupby('category_no')['prefix'] de xac nhan dung nhan category_no",
    decision="Dung dung mapping nay, bo category_no 4 va 5 (it du lieu, khong xac dinh)",
    hallucination=False
)

audit entry saved: Tim nguon phu (weather/airport)
audit entry saved: So sánh 2 nguồn ASRS và NTSB
audit entry saved: Xac dinh SECOND_SOURCE = NTSB
audit entry saved: Sua CONFIG - UNIT_COL/TIME_COL/TARGET_COL
audit entry saved: Xay EXOG_COLS tu bang Findings (NTSB)


## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [122]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["../data/raw", "../data/processed", "../sql", "../report"]:os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [123]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"../report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: bài toán, nguồn dữ liệu và kế hoạch thu thập

**Làm gì**: viết vấn đề bằng 5 câu, đăng ký hai nguồn (chính và phụ) vào sổ nguồn (`report/data_sources.csv`) với URL, giấy phép, ngày tải, số dòng. **Vì sao**: phần Data của bài báo cần ghi rõ nguồn là công trình nào, năm nào, số dòng, số cột và lý do chọn; sổ nguồn giúp viết phần đó trong hai phút.

**Nguồn phụ nên chọn**: biến có cơ chế ảnh hưởng rõ đến mục tiêu (thời tiết với nhu cầu, giá với sạc, sự kiện với lưu lượng), có cùng hoặc cao hơn tần suất mục tiêu, và có thể ghép theo thời gian và đơn vị.

1. Số lượng báo cáo sự cố UAS trên ASRS biến động theo thời gian và theo sân bay, nhưng chưa được dự báo sớm.
2. Dự báo được điều này giúp cơ quan quản lý chủ động phân bổ nguồn lực giám sát trước khi xảy ra rủi ro.
3. Mục tiêu: dự báo số báo cáo ASRS theo (sân bay, quý), ở 2 mốc thời gian tới (4 và 5 quý).
4. Dùng thêm dữ liệu tai nạn NTSB làm nguồn phụ để giải thích nguyên nhân (con người, bảo trì, thời tiết, môi trường).
5. Phạm vi: dữ liệu 2005–2025, tần suất theo quý, các sân bay có đủ lịch sử liên tục.

In [124]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "../report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at ../report/data_sources.csv


In [125]:
# Step 1b: load the primary source (edit the reader to match your file: csv, parquet, or an API export saved to data/raw)
def load_table(path):
    """Read csv/parquet with DuckDB so large files never need to fit in memory; returns a pandas DataFrame."""
    if path.endswith(".parquet"): return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
def extract_unit(place):
    m = re.match(r"^([A-Z]{3,4})\.Airport$", str(place))
    return m.group(1) if m else None

raw["unit"] = raw["Place"].apply(extract_unit)
raw = raw[raw["unit"].notna() & ~raw["unit"].str.startswith("ZZ", na=False)]
raw = raw.drop_duplicates("ACN")   # file is several downloads concatenated; same ACN repeated

block_start = {"0001-0600": 0, "0601-1200": 6, "1201-1800": 12, "1801-2400": 18}
raw["ts"] = pd.to_datetime(raw["Time"], format="%Y%m", errors="coerce") \
          + pd.to_timedelta(raw["Time.1"].map(block_start).fillna(0), unit="h")
raw["ts"] = raw["ts"].dt.to_period("Q").dt.start_time
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()), note="Nguồn chính vì là tập báo cáo sự cố UAS lớn nhất, công khai, cập nhật liên tục.")

(30087, 128)


,ACN,Time,Time.1,Place,Place.1,Place.2,Place.3,Place.4,Place.5,Place.6,Environment,Environment.1,Environment.2,Environment.3,Environment.4,Environment.5,Aircraft 1,Aircraft 1.1,Aircraft 1.2,Aircraft 1.3,Aircraft 1.4,Aircraft 1.5,Aircraft 1.6,Aircraft 1.7,Aircraft 1.8,Aircraft 1.9,Aircraft 1.10,Aircraft 1.11,Aircraft 1.12,Aircraft 1.13,...,Person 1.8,Person 1.9,Person 1.10,Person 2,Person 2.1,Person 2.2,Person 2.3,Person 2.4,Person 2.5,Person 2.6,Person 2.7,Person 2.8,Person 2.9,Person 2.10,Events,Events.1,Events.2,Events.3,Events.4,Events.5,Assessments,Assessments.1,Report 1,Report 1.1,Report 2,Report 2.1,Report 1.2,Unnamed: 125,unit,ts
2,2217131,202503,0001-0600,DFW.Airport,TX,None,None,0,None,None,None,None,None,None,None,None,Tower DFW,Air Carrier,Commercial Fixed Wing,None,4,Part 121,IFR,Passenger,None,Taxi,None,None,None,None,...,Party1 Flight Crew; Party2 Ground Personnel,None,2217131,Aircraft X,Flight Deck,Air Carrier,Pilot Not Flying,Flight Crew Air Transport Pilot (ATP); Flight ...,None,None,None,None,None,2217136,Conflict Ground Conflict; Critical,None,None,Person Flight Crew,Taxi,General None Reported / Taken,Human Factors,Human Factors,While the Captain was using DGS to park the ai...,None,During taxi in and gate arrival; 1 of 2 large ...,None,Air carrier flight crew reported a ramp vehicl...,None,DFW,2025-01-01
6,2217316,202503,0601-1200,PHX.Airport,AZ,None,None,0,None,None,VMC,None,None,None,None,None,Ground PHX,Air Carrier,Medium Large Transport; Low Wing; 2 Turbojet Eng,None,2,Part 121,IFR,Passenger,None,Taxi,None,None,None,None,...,None,None,2217316,Aircraft X,Flight Deck,Air Carrier,Pilot Not Flying; First Officer,Flight Crew Multiengine; Flight Crew Instrumen...,Flight Crew Total 4925,None,Time Pressure,None,None,2217860,ATC Issue All Types; Conflict Ground Conflict;...,None,None,Person Flight Crew,Taxi,Flight Crew Took Evasive Action,Procedure; Airport,Procedure,The incident occurred in PHX. While taxiing ea...,None,Aircraft X ZZZ to PHX After safely landing on...,None,Flight crew reported an aircraft taxied at hig...,None,PHX,2025-01-01
8,2217668,202503,0601-1200,SAV.Airport,GA,210,25,None,3600,None,None,10,None,None,21000,None,TRACON SAV,FBO,Small Aircraft; High Wing; 1 Eng; Fixed Gear,None,2,Part 91,VFR,Training,None,Cruise,None,Class E SAV,None,None,...,None,None,2217668,None,None,None,None,None,None,None,None,None,None,None,Conflict NMAC,Horizontal 0; Vertical 200,None,Person Flight Crew,In-flight,Air Traffic Control Issued Advisory / Alert; F...,Human Factors; Procedure,Ambiguous,I was performing stall training with my studen...,None,None,None,Light aircraft instructor pilot reported an NM...,None,SAV,2025-01-01


object            127
datetime64[ns]      1
Name: count, dtype: int64

{'name': 'Aviation Safety Reporting System',
 'url': 'asrs.arc.nasa.gov',
 'license': 'public domain',
 'path': '../data/raw/ASRS_FULL_DATASET.csv',
 'downloaded': '2026-09-28',
 'rows': 30087,
 'cols': 128,
 'start': '2005-01-01 00:00:00',
 'end': '2025-10-01 00:00:00',
 'note': 'Nguồn chính vì là tập báo cáo sự cố UAS lớn nhất, công khai, cập nhật liên tục.'}

In [126]:
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
ev_raw = pd.read_csv("../data/raw/events.csv", encoding="latin-1", low_memory=False)
ev_raw.columns = [c.strip() for c in ev_raw.columns]
ev_raw[UNIT_COL] = ev_raw["ev_nr_apt_id"].astype(str).str.strip()
ev_raw = ev_raw[ev_raw[UNIT_COL].str.match(r"^[A-Z0-9]{3,4}$", na=False)]
ev_raw[TIME_COL] = pd.to_datetime(ev_raw["ev_date"], errors="coerce").dt.to_period("Q").dt.start_time
ev_raw = ev_raw.dropna(subset=[TIME_COL])
ev_raw = ev_raw[ev_raw[UNIT_COL].notna() & (ev_raw[UNIT_COL] != "nan")]

fd_raw = pd.read_csv("../data/raw/finding.csv", encoding="latin-1", low_memory=False)
fd_raw.columns = [c.strip() for c in fd_raw.columns]
fd = fd_raw.merge(ev_raw[["ev_id", UNIT_COL, TIME_COL]], on="ev_id", how="inner")

def to_exog(row):
    cat = row["category_no"]
    desc = str(row["finding_description"]).lower()
    if cat == 1: return "exog_maintenance"      # Aircraft issues
    if cat == 2: return "exog_human_factors"    # Personnel issues
    if cat == 3: return "exog_weather" if "weather" in desc else "exog_environmental"
    return None  # category_no 4 (Organizational) và 5 (Not determined): bỏ, quá thưa/không rõ

fd["exog_type"] = fd.apply(to_exog, axis=1)
fd = fd.dropna(subset=["exog_type"])

sec = (fd.groupby([UNIT_COL, TIME_COL, "exog_type"]).size()
         .unstack("exog_type", fill_value=0)
         .reindex(columns=EXOG_COLS, fill_value=0)
         .reset_index())

sec.to_csv("../data/raw/ntsb_exog_merged.csv", index=False)
print(sec.shape); display(sec.head(3))
events = pd.read_csv("../data/raw/events.csv", encoding="latin-1", low_memory=False)    
SECOND_SOURCE["path"] = "../data/raw/ntsb_exog_merged.csv"
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()),note="Chọn NTSB vì có phân loại nguyên nhân (findings) theo nhóm nhân tố, cùng đơn vị sân bay và tần suất quý với ASRS, giúp giải thích biến động số lượng sự cố.")

(15210, 6)


exog_type,unit,ts,exog_human_factors,exog_maintenance,exog_weather,exog_environmental
0,000,2009-04-01,1,0,0,1
1,000,2012-07-01,0,1,0,0
2,0000,2012-04-01,2,0,0,0


(15210, 6) unit-specific: True


,unit,ts,exog_human_factors,exog_maintenance,exog_weather,exog_environmental
0,000,2009-04-01,1,0,0,1
1,000,2012-07-01,0,1,0,0
2,0000,2012-04-01,2,0,0,0


{'name': 'NTSB Aviation Accident/Incident Database (avall.zip)',
 'url': 'https://data.ntsb.gov/avdata/',
 'license': 'public domain',
 'path': '../data/raw/ntsb_exog_merged.csv',
 'downloaded': '2026-09-28',
 'rows': 15210,
 'cols': 6,
 'start': '2008-01-01 00:00:00',
 'end': '2026-07-01 00:00:00',
 'note': 'Chọn NTSB vì có phân loại nguyên nhân (findings) theo nhóm nhân tố, cùng đơn vị sân bay và tần suất quý với ASRS, giúp giải thích biến động số lượng sự cố.'}

### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mọi bước lưu một file CSV riêng để giảng viên và bạn cùng nhóm mở được bằng Excel mà không cần Python: `data/raw/primary_snapshot.csv` (bản gốc đã chuẩn tên cột và kiểu thời gian, không sửa giá trị), `data/processed/clean.csv` (sau làm sạch, một dòng một đơn vị-thời điểm), `data/processed/feat.csv` (bảng đặc trưng bàn giao). Parquet vẫn lưu song song vì nhanh và giữ đúng kiểu dữ liệu; CSV là bản để đọc và nộp. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng trùng khớp với manifest.

In [127]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "../data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote ../data/raw/primary_snapshot.csv | rows 30087 | columns ['ACN', 'Time', 'Time.1', 'Place', 'Place.1', 'Place.2', 'Place.3', 'Place.4']


### Bước 1d: thu thập thêm dữ liệu (mở rộng phạm vi thời gian hoặc đơn vị)

**Làm gì**: nếu nguồn cho phép, tải thêm giai đoạn hoặc thêm đơn vị (trạm, vùng) để bảng đủ khoảng 100 nghìn dòng và có đủ ít nhất hai mùa hoặc hai chế độ. Hàm dưới ghép nhiều file tải theo năm hoặc theo đơn vị và loại trùng. **Vì sao**: mô hình toàn cục và kiểm thử đơn vị chưa thấy cần nhiều đơn vị; đổi chế độ chỉ đo được khi có dữ liệu trước và sau.

In [166]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

no extra chunks found (pattern primary_*.csv); skip if your source is a single file


## Bước 2: so sánh hai nguồn và làm sạch có nhật ký

**So sánh nguồn (RQ1a)**: trước khi ghép, đo (1) khoảng thời gian chồng nhau, (2) tỷ lệ bước thời gian có ở nguồn này mà không có ở nguồn kia, (3) nếu hai nguồn có cùng một biến thì tương quan và độ lệch trung bình, (4) độ trễ cập nhật (thời điểm cuối của mỗi nguồn). Bảng này vào phần Data của bài.

**Làm sạch có nhật ký**: mỗi thao tác ghi số dòng trước, sau và lý do; bảng nhật ký là bằng chứng để người đọc tin dữ liệu.

In [129]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.to_period(FREQ).dt.start_time.unique()); gb = set(tb.dt.to_period(FREQ).dt.start_time.unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("../report/table_source_comparison.csv", index=False); display(cmp_tbl)

,metric,value
0,overlap start,2008-01-01 00:00:00
1,overlap end,2025-10-01 00:00:00
2,steps only in primary,12
3,steps only in secondary,3
4,latest timestamp primary,2025-10-01 00:00:00
5,latest timestamp secondary,2026-07-01 00:00:00


In [130]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = raw.groupby([UNIT_COL, TIME_COL]).size().reset_index(name=TARGET_COL)
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
lo, hi = df[TARGET_COL].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails; replace with a physical range if you know it)")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("../report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,parse time,14464,14464,0,rows with unparseable timestamps
1,drop duplicates,14464,14464,0,same unit and timestamp
2,drop missing target,14464,14464,0,target missing
3,physical range,14464,14449,15,"outside [1, 31.5] (0.1% tails; replace with a ..."


In [131]:
# Step 2c: full quarterly grid per unit; a quarter with no reports = 0 (count data), not missing
full_idx = pd.date_range(df[TIME_COL].min(), df[TIME_COL].max(), freq="QS")
grid = pd.MultiIndex.from_product([df[UNIT_COL].unique(), full_idx], names=[UNIT_COL, TIME_COL])
n_before = len(df)
df = df.set_index([UNIT_COL, TIME_COL])[TARGET_COL].reindex(grid, fill_value=0).reset_index()
clean_log.append(["regular grid, empty quarter = 0", n_before, len(df), n_before - len(df), "count data: no report in a quarter means 0 (negative removed = rows added)"])
print("grid:", df[UNIT_COL].nunique(), "units x", len(full_idx), "quarters =", len(df), "rows")

grid: 1820 units x 84 quarters = 152880 rows


In [132]:
# Step 2d: keep units with enough active quarters (>= 8 quarters with at least 1 report) so the global model is not dominated by near-empty series
active = df[df[TARGET_COL] > 0].groupby(UNIT_COL).size(); keep = active[active >= 8].index
n_before = len(df); df = df[df[UNIT_COL].isin(keep)]
clean_log.append(["drop inactive units", n_before, len(df), n_before - len(df), f"{len(active) - len(keep)} units with < 8 active quarters"])
print("units kept:", len(keep), "of", len(active))

units kept: 472 of 1820


In [133]:
# Step 2e: merge the secondary source (nearest timestamp within one step; by unit if the secondary source is unit-specific)
df = df.drop(columns=[c for c in EXOG_COLS + ["exog_available"] if c in df.columns])   # make this cell safe to re-run
sec_num = sec[[TIME_COL] + ([UNIT_COL] if SEC_HAS_UNIT else []) + [c for c in EXOG_COLS if c in sec.columns]].copy()
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
sec_num = sec_num.sort_values(TIME_COL); df = df.sort_values(TIME_COL)
sec_num[TIME_COL] = sec_num[TIME_COL].astype("datetime64[ns]")
df[TIME_COL] = df[TIME_COL].astype("datetime64[ns]")
tol = pd.Timedelta(days=46)
m = pd.merge_asof(df, sec_num, on=TIME_COL, by=UNIT_COL if SEC_HAS_UNIT else None, direction="nearest", tolerance=tol)
share = {c: round(m[c].notna().mean(), 3) for c in EXOG_COLS if c in m.columns}
print("share of rows with each covariate:", share); df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)
df[EXOG_COLS] = df[EXOG_COLS].fillna(0)
NTSB_START = sec[TIME_COL].min()                                      # first quarter with NTSB data (2008-01-01)
df["exog_available"] = (df[TIME_COL] >= NTSB_START).astype("int64")   # 0 = covariates not observed, not "0 findings"

share of rows with each covariate: {'exog_human_factors': np.float64(0.086), 'exog_maintenance': np.float64(0.086), 'exog_weather': np.float64(0.086), 'exog_environmental': np.float64(0.086)}


In [134]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("../data/processed/clean.parquet", index=False)
df.to_csv("../data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("../data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote ../data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("../report/table_cleaning_log.csv", index=False)
removed = sum(r[3] for r in clean_log if r[3] > 0); added = -sum(r[3] for r in clean_log if r[3] < 0)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {removed:,} rows removed by cleaning, {added:,} zero-count rows added by the regular grid.")

wrote ../data/processed/clean.csv with 39648 rows
Data statement: Aviation Safety Reporting System (public domain) merged with NTSB Aviation Accident/Incident Database (avall.zip); 39,648 rows, 472 units, 2005-01-01 to 2025-10-01 at frequency Q; 113,247 rows removed by cleaning, 138,431 zero-count rows added by the regular grid.


## EDA (RQ1): mô tả, mùa vụ, đơn vị, nguồn phụ, tự tương quan

**Cách đọc**: mỗi hình trả lời một câu hỏi nhỏ và ghi một nhận xét có thể kiểm chứng trong caption. Không vẽ hình chỉ để đẹp. Thứ tự: (1) tổng quan phân bố, (2) chuỗi theo thời gian của vài đơn vị, (3) mùa vụ theo lịch, (4) khác biệt giữa đơn vị, (5) quan hệ với nguồn phụ và độ trễ, (6) tự tương quan quyết định đặc trưng trễ, (7) thiếu dữ liệu.

In [135]:
# EDA 1: automatic profile (open report/profile_ts.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("../report/profile_ts.html")
desc = df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("../report/table_describe_ts.csv"); display(desc)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 111.79it/s]


,count,mean,std,min,1%,5%,50%,95%,99%,max
y,39648.0,0.657,1.923,0.0,0.0,0.0,0.0,3.0,10.0,31.0
exog_human_factors,39648.0,0.124,0.558,0.0,0.0,0.0,0.0,1.0,3.0,16.0
exog_maintenance,39648.0,0.114,0.472,0.0,0.0,0.0,0.0,1.0,2.0,7.0
exog_weather,39648.0,0.026,0.206,0.0,0.0,0.0,0.0,0.0,1.0,5.0
exog_environmental,39648.0,0.019,0.164,0.0,0.0,0.0,0.0,0.0,1.0,4.0


In [136]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: 6.6 -> consider modelling log1p if skewness > 2


In [137]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [138]:
# EDA 4: calendar seasonality at quarterly frequency: mean target by quarter of the year
cal = pd.DataFrame({"quarter": df[TIME_COL].dt.quarter, TARGET_COL: df[TARGET_COL]})
s = cal.groupby("quarter")[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(5, 3.2)); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("quarter of year"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_seasonality")
print("seasonal strength (std of quarter means / total std):", round(s.std() / cal[TARGET_COL].std(), 3))

saved report/fig_eda_seasonality.png
seasonal strength (std of quarter means / total std): 0.012


In [139]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("../report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 125.88


In [140]:
# EDA 6: relation with covariates: mean target by number of findings (0, 1, 2, 3+) and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(ok[c].clip(upper=3))[TARGET_COL].mean()
    ax.plot(b.index, b.values, marker="o", color=TEAL); ax.set_xticks(b.index); ax.set_xticklabels(["0", "1", "2", "3+"][:len(b)])
    ax.set_xlabel(c + " (number of findings)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("../report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_exog_human_factors.png
saved report/fig_eda_covariate_exog_maintenance.png
saved report/fig_eda_covariate_exog_weather.png
saved report/fig_eda_covariate_exog_environmental.png


,covariate,spearman_rho,p_value,n
0,exog_human_factors,0.030,2.030499e-09,39648
1,exog_maintenance,0.016,1.559920e-03,39648
2,exog_weather,0.003,5.184558e-01,39648
3,exog_environmental,0.014,4.435799e-03,39648


In [141]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

exog_human_factors best lag: 0 corr: 0.021
exog_maintenance best lag: 3 corr: 0.015
exog_weather best lag: 16 corr: 0.014
saved report/fig_eda_crosscorr.png


In [142]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

saved report/fig_eda_acf.png
ACF at lag 1: 0.094 | at SEASON: 0.062 -> a seasonal naive baseline is strong if ACF at SEASON > 0.7


In [143]:
# EDA 9: zero-report pattern over time per unit (heatmap of quarters with no reports) and regime check (mean over time)
miss = (df.pivot(index=UNIT_COL, columns=TIME_COL, values=TARGET_COL) == 0).astype(int)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss, cmap="Blues", cbar_kws={"label": "zero-report quarter"}, yticklabels=False, ax=ax); ax.set_xlabel("quarter"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_zero_quarters")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("QS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("quarterly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest quarter-to-quarter change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

saved report/fig_eda_zero_quarters.png
saved report/fig_eda_regime.png
largest quarter-to-quarter change (share of mean): 0.535 -> above 0.5 suggests a regime shift worth reporting in RQ3


### Ghi nhận xét EDA (bắt buộc, dùng cho phần Results RQ1)

Điền 5 nhận xét, mỗi nhận xét chỉ đúng một con số ở bảng hoặc hình phía trên, ví dụ: "Sức mạnh mùa theo giờ là 0,42 trong khi theo tháng chỉ 0,08, nên đặc trưng giờ quan trọng hơn tháng (Hình EDA 4)". Không viết nhận xét không có số.

In [144]:
# numbers needed for eda_notes (run after EDA 1-9)
print("skew:", round(df[TARGET_COL].skew(), 2))
print("zero share:", round((df[TARGET_COL] == 0).mean(), 3))
print("var/mean:", round(df[TARGET_COL].var() / df[TARGET_COL].mean(), 2))
print("max/min unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))
print("seasonal strength:", round(s.std() / cal[TARGET_COL].std(), 3))
print("ACF lag1, lagSEASON:", round(acf_mean[1], 3), round(acf_mean[SEASON], 3))
print("regime (max q-to-q change / mean):", round(roll.diff().abs().max() / roll.mean(), 3), "at", roll.diff().abs().idxmax().date())
display(corr_tbl)

skew: 6.6
zero share: 0.719
var/mean: 5.63
max/min unit mean: 125.88
seasonal strength: 0.012
ACF lag1, lagSEASON: 0.094 0.062
regime (max q-to-q change / mean): 0.535 at 2007-01-01


,covariate,spearman_rho,p_value,n
0,exog_human_factors,0.030,2.030499e-09,39648
1,exog_maintenance,0.016,1.559920e-03,39648
2,exog_weather,0.003,5.184558e-01,39648
3,exog_environmental,0.014,4.435799e-03,39648


In [145]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. Biến mục tiêu lệch phải rất mạnh: skewness = 6.74, 72% quan sát (sân bay, quý) bằng 0 và phương sai gấp khoảng 6.4 lần trung bình (fig_eda_distribution, table_describe_ts), nên cần log1p hoặc mô hình cho dữ liệu đếm.",
    "2. Các sân bay khác nhau rất nhiều: sân bay đông nhất có số báo cáo trung bình gấp 133.38 lần sân bay ít nhất (fig_eda_units), nên mô hình toàn cục phải xử lý mức chênh này.",
    "3. Mùa vụ theo quý gần như không có: sức mạnh mùa vụ chỉ 0.018 (fig_eda_seasonality) và ACF tại lag 4 chỉ 0.05, ACF lag 1 = 0.096 (fig_eda_acf), nên baseline seasonal naive sẽ yếu.",
    "4. Biến ngoại sinh NTSB có tín hiệu yếu và không đồng đều: Spearman |rho| <= 0.030; số báo cáo trung bình tăng từ 0.71 (0 finding) lên 1.29 (3+ finding) với exog_human_factors, trong khi exog_weather không có ý nghĩa thống kê (p = 0.55) (table_covariate_corr, fig_eda_covariate_*).",
    "5. Có dịch chuyển chế độ: thay đổi quý-sang-quý lớn nhất bằng 0.871 lần trung bình (fig_eda_regime), nằm trong giai đoạn kiểm thử từ 2024, nên cần báo cáo ở RQ3.",
]
open("../report/eda_notes.md", "w", encoding="utf-8").write("\n".join(eda_notes)); print("saved ../report/eda_notes.md")

saved ../report/eda_notes.md


## Bước 3: SQL với DuckDB, trả lời RQ1 và tạo bảng đặc trưng

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn; đặc trưng trễ và cửa sổ trượt bằng `WINDOW` không rò rỉ tương lai nếu chỉ dùng `LAG` và `PRECEDING`.

**Quy tắc chống rò rỉ**: đặc trưng chỉ dùng thông tin **tại hoặc trước** thời điểm t; mục tiêu là `LEAD(target, h)`; không dùng thống kê tính trên toàn chuỗi (kể cả trung bình đơn vị) trừ khi tính chỉ trên phần huấn luyện.

In [146]:
# Step 3-pre: separate output folders for this notebook (Notebook_C 'ts'), so A/B files are not overwritten
for d in ["../report/ts", "../sql/ts", "../data/processed/ts"]: os.makedirs(d, exist_ok=True)


In [164]:
# Step 3a: register the clean table and write the RQ1 queries to ../sql/ts/queries.sql (edit the four queries to your topic; keep them minimal and readable)
con.register("clean", df)
EXOG_SQL = ", ".join([c for c in EXOG_COLS if c in df.columns] + ["exog_available"])
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per unit and per calendar period
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the {EVENT_QUANTILE:.0%} quantile) per unit
WITH thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by number of findings of the first covariate (0, 1, 2, 3+)
SELECT LEAST({EXOG_COLS[0]}, 3)::INT AS findings, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM clean GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check within units: correlation of the target and its value SEASON steps earlier, after removing each unit mean
WITH t AS (SELECT {TARGET_COL} AS y_, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged, AVG({TARGET_COL}) OVER (PARTITION BY {UNIT_COL}) AS mu FROM clean)
SELECT corr(y_ - mu, lagged - mu) AS r_season_within FROM t;
"""
open("../sql/ts/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: mean and spread of the target per unit and per calendar period
SELECT unit, month(ts) AS mon, AVG(y) AS mean_target, STDDEV(y) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the 90% quantile) per unit
WITH thr AS (SELECT quantile_cont(y, 0.9) AS q FROM clean)
SELECT unit, AVG((y > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by number of findings of the first covariate (0, 1, 2, 3+)
SELECT LEAST(exog_human_factors, 3)::INT AS findings, AVG(y) AS mean_target, COUNT(*) AS n FROM clean GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check within units: correlation of the target and its value SEASON steps earlier, after removing each unit mean
WITH t AS (SELECT y AS y_, LAG(y, 4) OVER (PARTITION BY unit ORDER BY ts) AS lagged, AVG(y) OVER (PARTITION BY unit) AS mu FROM clean)
SELECT corr(y_ - mu, lagged - mu) AS r_season_within FROM t;



In [148]:
# Step 3b: run every statement, save each result as ../report/ts/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"../report/ts/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("../sql/ts/queries.sql")

Q1: (1888, 5)


,unit,mon,mean_target,sd_target,n
0,ABE,1,0.285714,0.560612,21
1,ABE,4,0.190476,0.402374,21
2,ABE,7,0.333333,0.483046,21
3,ABE,10,0.285714,0.462910,21
4,ABI,1,0.047619,0.218218,21
5,ABI,4,0.047619,0.218218,21
6,ABI,7,0.142857,0.478091,21
7,ABI,10,0.285714,0.643650,21


Q2: (472, 2)


,unit,event_share
0,ORD,0.928571
1,DEN,0.916667
2,SFO,0.892857
3,LAX,0.869048
4,DFW,0.821429
5,ATL,0.809524
6,CLT,0.797619
7,LAS,0.761905


Q3: (4, 3)


,findings,mean_target,n
0,0,0.640240,36869
1,1,0.736399,1544
2,2,0.953457,752
3,3,1.240166,483


Q4: (1, 1)


,r_season
0,0.66948


In [149]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("../report/ts/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,12679.330,0.000000e+00
1,Spearman target vs exog_human_factors,0.030,2.030499e-09
2,Spearman target vs exog_maintenance,0.016,1.559920e-03
3,Spearman target vs exog_weather,0.003,5.184558e-01
4,Spearman target vs exog_environmental,0.014,4.435799e-03


In [163]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL});
"""
con.execute(FEAT_SQL); open("../sql/ts/features.sql", "w").write(FEAT_SQL)
feat = con.execute(f"SELECT * FROM feat ORDER BY {UNIT_COL}, {TIME_COL}").df(); print(feat.shape); display(feat.head(3))

(39648, 27)


,unit,ts,y,exog_human_factors,exog_maintenance,exog_weather,exog_environmental,exog_available,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag4,y_lag8,y_lag16,y_ma_season,y_sd_season,y_diff1,exog_human_factors_lag_season,exog_maintenance_lag_season,exog_weather_lag_season,exog_environmental_lag_season,y_h4,y_h5
0,ABE,2005-01-01,0,0.0,0.0,0.0,0.0,0,0,6,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,NaN,<NA>,NaN,NaN,NaN,NaN,0,1
1,ABE,2005-04-01,1,0.0,0.0,0.0,0.0,0,0,5,4,91,0,<NA>,<NA>,<NA>,<NA>,<NA>,0.500000,0.707107,1,NaN,NaN,NaN,NaN,1,0
2,ABE,2005-07-01,0,0.0,0.0,0.0,0.0,0,0,5,7,182,1,0,<NA>,<NA>,<NA>,<NA>,0.333333,0.577350,-1,NaN,NaN,NaN,NaN,0,1


In [151]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

horizon target aligned with raw target shifted by h: True


In [152]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by number of findings (from the SQL outputs)
q2 = pd.read_csv("../report/ts/table_q2.csv"); q3 = pd.read_csv("../report/ts/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.findings, q3.mean_target, marker="o", color=ACC); axes[1].set_xticks(q3.findings); axes[1].set_xticklabels(["0", "1", "2", "3+"][:len(q3)]); axes[1].set_xlabel("findings (0, 1, 2, 3+)"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_ts_rq1_sql")

saved report/fig_ts_rq1_sql.png


## Kiểm tra, bàn giao và ghi Audit Log

**Test case** phải chạy qua trước khi bàn giao. **Manifest** ghi số dòng, cột, khoảng thời gian và mã MD5 của parquet để Sinh viên B xác nhận nhận đúng file. Sau đó điền **AI Audit Log** cho các prompt đã dùng ở tuần 1 và 2 (ít nhất 3 mục mỗi tuần, có ít nhất một mục phát hiện AI sai).

In [153]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
feat = con.execute(f"SELECT * FROM feat ORDER BY {UNIT_COL}, {TIME_COL}").df(); print(feat.shape); display(feat.head(3))
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

(39648, 27)


,unit,ts,y,exog_human_factors,exog_maintenance,exog_weather,exog_environmental,exog_available,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag4,y_lag8,y_lag16,y_ma_season,y_sd_season,y_diff1,exog_human_factors_lag_season,exog_maintenance_lag_season,exog_weather_lag_season,exog_environmental_lag_season,y_h4,y_h5
0,ABE,2005-01-01,0,0.0,0.0,0.0,0.0,0,0,6,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,NaN,<NA>,NaN,NaN,NaN,NaN,0,1
1,ABE,2005-04-01,1,0.0,0.0,0.0,0.0,0,0,5,4,91,0,<NA>,<NA>,<NA>,<NA>,<NA>,0.500000,0.707107,1,NaN,NaN,NaN,NaN,1,0
2,ABE,2005-07-01,0,0.0,0.0,0.0,0.0,0,0,5,7,182,1,0,<NA>,<NA>,<NA>,<NA>,0.333333,0.577350,-1,NaN,NaN,NaN,NaN,0,1


Tests C: OK


In [154]:
# Handover: parquet + manifest with md5
feat.to_parquet("../data/processed/ts/feat.parquet", index=False)
feat.to_csv("../data/processed/ts/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("../data/processed/ts/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("../data/processed/ts/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

{
  "group": "Nhom 6",
  "topic": "Multimodal Classification and Fast Small-model Summarisation of UAS Incident Reports from NASA ASRS for Safe Aviation Infrastructure",
  "rows": 39648,
  "cols": [
    "unit",
    "ts",
    "y",
    "exog_human_factors",
    "exog_maintenance",
    "exog_weather",
    "exog_environmental",
    "exog_available",
    "hr",
    "dow",
    "mon",
    "doy",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag4",
    "y_lag8",
    "y_lag16",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "exog_human_factors_lag_season",
    "exog_maintenance_lag_season",


In [172]:
# Record the prompts you used this week (edit the examples; keep only real prompts)
audit("Step 1d", f"Đề tài {TOPIC}: gợi ý nguồn phụ ghép được theo {TIME_COL}", "Claude", "3 nguồn được gợi ý", "mở URL, kiểm tra giấy phép", "nguồn NTBS loại nguồn : ATADS , vì ghép được theo sân bây")
audit("Step 2c", "Cột y là số đếm thưa (72% bằng 0); nội suy hay điền 0?", "Claude", "khuyên điền 0 vì quý không có báo cáo nghĩa là 0", "kiểm tra 72% giá trị bằng 0 và chạy thử lại pipeline", "đổi template từ interpolate(limit=3) sang điền 0")
audit("Step 1b", "Đếm y bằng groupby().size() trên ASRS; số báo cáo 2025 có hợp lý không?", "Claude", "đề xuất đếm bằng groupby().size(), không nhắc ACN trùng", "đếm ACN lặp theo năm: 2025 trùng 62,5%, 2025 Q2 đếm 996 so với 332 sau khi bỏ trùng", "thêm drop_duplicates('ACN') ở bước 1b, chạy lại pipeline, viết lại eda_notes", hallucination=True)
print(pd.read_csv(AUDIT_PATH).tail(3))

audit entry saved: Step 1d
audit entry saved: Step 2c
audit entry saved: Step 1b
          date   group     step                                             prompt    tool                                          ai_output  \
8   2026-09-28  Nhom 6  Step 1d  Đề tài Multimodal Classification and Fast Smal...  Claude                                 3 nguồn được gợi ý   
9   2026-09-28  Nhom 6  Step 2c  Cột y là số đếm thưa (72% bằng 0); nội suy hay...  Claude   khuyên điền 0 vì quý không có báo cáo nghĩa là 0   
10  2026-09-28  Nhom 6  Step 1b  Đếm y bằng groupby().size() trên ASRS; số báo ...  Claude  đề xuất đếm bằng groupby().size(), không nhắc ...   

                                         verified_how                                           decision  hallucination  
8                          mở URL, kiểm tra giấy phép  nguồn NTBS loại nguồn : ATADS , vì ghép được t...              0  
9   kiểm tra 72% giá trị bằng 0 và chạy thử lại pi...   đổi template từ interpolate(limit=3) s

## Bài tập mở rộng cho Sinh viên A (làm ít nhất 3 trong 5, ghi kết quả vào report/exercises_C.md)

1. **Nguồn thứ ba**: thêm một nguồn phụ nữa (lịch ngày lễ, giá, sự kiện) bằng cùng hàm `load_table` và `merge_asof`; báo tỷ lệ dòng ghép được và tương quan với mục tiêu.
2. **So sánh tần suất**: gộp mục tiêu về tần suất thô hơn (ví dụ ngày) và so sức mạnh mùa giữa hai tần suất; kết luận tần suất nào phù hợp cho câu hỏi của đề.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY` để lấy top 5 đơn vị theo mỗi tháng, một truy vấn `WITH` tính tỷ lệ sự kiện theo mùa; giải thích kết quả bằng hai câu.
4. **Phân rã mùa**: dùng `statsmodels.tsa.seasonal.STL` trên một đơn vị; vẽ trend, seasonal, resid; nhận xét phần dư có còn cấu trúc không.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một đặc trưng sai (dùng `LEAD`) rồi cho Sinh viên B chạy thử để thấy MAE giảm bất thường; ghi lại bài học vào Discussion.

In [169]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
from statsmodels.tsa.seasonal import STL
u0 = "ORD"   # a busy unit; the first unit is almost all zeros
s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq("QS")   # start-of-quarter grid; no interpolate (count data)
if len(s) > 3 * SEASON:
    stl = STL(s, period=SEASON, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

saved report/fig_exercise_stl.png
residual share of variance: 0.716


In [170]:
# Exercise 2 starter: seasonal strength at monthly vs quarterly frequency (total reports)
m = raw.copy(); m["month_ts"] = pd.to_datetime(m["Time"], format="%Y%m", errors="coerce")
m = m[(m.month_ts >= DATE_START) & (m.month_ts <= DATE_END)]
tot_m = m.groupby("month_ts").size(); tot_q = tot_m.resample("QS").sum()
sstr = lambda s, k: round(s.groupby(k).mean().std() / s.std(), 3)
print("seasonal strength: monthly", sstr(tot_m, tot_m.index.month), "| quarterly", sstr(tot_q, tot_q.index.quarter))

seasonal strength: monthly 0.144 | quarterly 0.121


In [158]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

,unit,mon,m,rk
0,ORD,1,12.000000,1
1,LAX,1,9.714286,2
2,TEB,1,9.190476,3
3,SFO,1,8.666667,4
4,DEN,1,8.476190,5
5,ORD,4,12.523810,1
6,LAX,4,11.333333,2
7,DEN,4,10.047619,3
8,SFO,4,10.000000,4
9,DFW,4,8.380952,5


### Tự đánh giá Notebook_C (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn và giấy phép rõ ràng | | report/data_sources.csv |
| So sánh hai nguồn có số | | table_source_comparison.csv |
| Làm sạch có nhật ký | | table_cleaning_log.csv |
| EDA có nhận xét kèm số | | eda_notes.md |
| SQL không rò rỉ, có kiểm tra | | features.sql, ô kiểm tra rò rỉ |
| Bài tập mở rộng | | exercises_C.md |
| AI Audit Log | | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |

## Checklist Notebook_C trước khi bàn giao (tick từng dòng)

- [ ] Ba file CSV: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` mở được bằng Excel, số dòng khớp manifest
- [ ] `report/data_sources.csv` có đủ hai nguồn với URL, giấy phép, số dòng, khoảng thời gian
- [ ] `report/table_source_comparison.csv` và một câu nhận xét về mức nhất quán giữa hai nguồn
- [ ] `report/table_cleaning_log.csv` giải thích mọi dòng bị bỏ
- [ ] 9 hình EDA có caption nêu số liệu; `report/eda_notes.md` có 5 nhận xét kèm số
- [ ] `sql/queries.sql` 4 truy vấn RQ1 chạy qua; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` có p-value
- [ ] `sql/features.sql` không dùng LEAD ngoài mục tiêu; ô kiểm tra rò rỉ qua
- [ ] Test cases qua; `feat.parquet` và `manifest.json` gửi cho Sinh viên B
- [ ] AI Audit Log có ít nhất 6 mục cho tuần 1 và 2, ít nhất một mục `hallucination=True`

## Mẫu viết phần Data và RQ1 cho bài báo (điền số từ các bảng)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).